In [ ]:
import pandas as pd
import requests
import time
from base64 import b64encode

# Spotify API credentials - you'll need to get these from https://developer.spotify.com/dashboard
CLIENT_ID = 'XXXXXXXXXXXX'
CLIENT_SECRET = 'XXXXXXXXXXX'

# File paths
input_file = '/Users/shannon/COMM 557/COMM 557 - Final Project/data/UPDATED_dataset_with_topics_communities_crossover.csv'
output_file = '/Users/shannon/COMM 557/COMM 557 - Final Project/data/UPDATED_dataset_with_artist_pop.csv'

def get_spotify_token(client_id, client_secret):
    """Get Spotify API access token"""
    auth_string = f"{client_id}:{client_secret}"
    auth_bytes = auth_string.encode("utf-8")
    auth_base64 = b64encode(auth_bytes).decode("utf-8")
    
    url = "https://accounts.spotify.com/api/token"
    headers = {
        "Authorization": f"Basic {auth_base64}",
        "Content-Type": "application/x-www-form-urlencoded"
    }
    data = {"grant_type": "client_credentials"}
    
    response = requests.post(url, headers=headers, data=data)
    return response.json()["access_token"]

def search_artist_info(artist_name, token):
    """Search for an artist and return their popularity and main genre"""
    url = "https://api.spotify.com/v1/search"
    headers = {"Authorization": f"Bearer {token}"}
    params = {
        "q": artist_name,
        "type": "artist",
        "limit": 1
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
        
        if data['artists']['items']:
            artist = data['artists']['items'][0]
            popularity = artist['popularity']
            # Get the first (main) genre if available
            main_genre = artist['genres'][0] if artist['genres'] else None
            return {'popularity': popularity, 'genre': main_genre}
        else:
            return None
    except Exception as e:
        print(f"Error searching for {artist_name}: {e}")
        return None

def get_max_artist_info(artist_name_str, token):
    """
    Split artist names by comma, get info for each,
    return the popularity and genre from the most popular artist
    """
    if pd.isna(artist_name_str):
        return {'popularity': None, 'genre': None}
    
    # Split by comma and clean up whitespace
    artists = [a.strip() for a in str(artist_name_str).split(',')]
    
    artist_infos = []
    for artist in artists:
        if artist:  # Skip empty strings
            info = search_artist_info(artist, token)
            if info is not None:
                artist_infos.append(info)
            time.sleep(0.1)  # Small delay to avoid rate limiting
    
    if not artist_infos:
        return {'popularity': None, 'genre': None}
    
    # Find the artist with max popularity
    max_artist = max(artist_infos, key=lambda x: x['popularity'])
    
    return {
        'popularity': max_artist['popularity'],
        'genre': max_artist['genre']
    }

# Main execution
print("Getting Spotify API token...")
token = get_spotify_token(CLIENT_ID, CLIENT_SECRET)
print("Token obtained!")

# Read the dataset
print("\nReading dataset...")
df = pd.read_csv(input_file)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Get unique artists to minimize API calls
print("\nExtracting unique artists...")
unique_artists = df['artist_name'].unique()
print(f"Found {len(unique_artists)} unique artist combinations")

# Create a dictionary to store artist info
artist_info_dict = {}

print("\nFetching artist info from Spotify API...")
print("This may take a while...")

for i, artist_combo in enumerate(unique_artists):
    if i % 10 == 0:  # Progress update every 10 artists
        print(f"Progress: {i}/{len(unique_artists)}")
    
    artist_info_dict[artist_combo] = get_max_artist_info(artist_combo, token)
    
    # Refresh token every 500 requests (tokens expire after ~1 hour)
    if i % 500 == 0 and i > 0:
        print("Refreshing token...")
        token = get_spotify_token(CLIENT_ID, CLIENT_SECRET)

print(f"\nCompleted! Fetched info for {len(artist_info_dict)} artist combinations")

# Map the info back to the dataframe
df['max_artist_popularity'] = df['artist_name'].map(lambda x: artist_info_dict.get(x, {}).get('popularity'))
df['main_artist_genre'] = df['artist_name'].map(lambda x: artist_info_dict.get(x, {}).get('genre'))

# Show statistics
print(f"\nResults:")
print(f"Total rows: {len(df)}")
print(f"Rows with max_artist_popularity: {df['max_artist_popularity'].notna().sum()}")
print(f"Rows without max_artist_popularity: {df['max_artist_popularity'].isna().sum()}")
print(f"Rows with main_artist_genre: {df['main_artist_genre'].notna().sum()}")
print(f"Rows without main_artist_genre: {df['main_artist_genre'].isna().sum()}")
print(f"\nArtist popularity stats:")
print(df['max_artist_popularity'].describe())
print(f"\nTop 10 genres:")
print(df['main_artist_genre'].value_counts().head(10))

# Show sample with highest popularity artists
print("\nTop 10 highest popularity artists:")
print(df[['artist_name', 'max_artist_popularity', 'main_artist_genre']].drop_duplicates().sort_values('max_artist_popularity', ascending=False).head(10))

print("\n" + "="*50)
print("DATA READY! Check the dataframe with df.head()")
print("When ready to save, run the next cell")
print("="*50)

Getting Spotify API token...
Token obtained!

Reading dataset...
Dataset shape: (4521, 31)
Columns: ['track_name', 'artist_name', 'danceability', 'energy', 'loudness', 'mode', 'key', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'tempo', 'time_signature', 'duration_ms', 'source', 'lyrics', 'lyrics_missing', 'topic_number', 'topic_label', 'community', 'label_short', 'label_long', 'category', 'community_simplified', 'label_short_simplified', 'label_long_simplified', 'category_simplified', 'track_type', 'is_cross', 'year', 'chart_longevity']

Extracting unique artists...
Found 1918 unique artist combinations

Fetching artist info from Spotify API...
This may take a while...
Progress: 0/1918
Progress: 10/1918
Progress: 20/1918
Progress: 30/1918
Progress: 40/1918
Progress: 50/1918
Progress: 60/1918
Progress: 70/1918
Progress: 80/1918
Progress: 90/1918
Progress: 100/1918
Progress: 110/1918
Progress: 120/1918
Progress: 130/1918
Progress: 140/1918
Progress: 150/1918
Progress: 

In [3]:
# Save to CSV
output_file = '/Users/shannon/COMM 557/COMM 557 - Final Project/data/UPDATED_dataset_with_artist_pop.csv'
df.to_csv(output_file, index=False)
print(f"Saved updated dataset to: {output_file}")

Saved updated dataset to: /Users/shannon/COMM 557/COMM 557 - Final Project/data/UPDATED_dataset_with_artist_pop.csv
